# Banking agent: VAE + GAN + LoRA (PEFT) on Google Colab

Runs the three new modules. Use a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU).

1. Setup (upload the project, install packages)
2. VAE + GAN experiment (`src.augment`) - works on CPU or GPU
3. LoRA fine-tuning of Qwen2.5-1.5B-Instruct (`src.peft_lora`) - needs the GPU
4. Compare RAG vs PEFT vs hybrid on the held-out subset
5. Optional: RAG with the synthetic pool, and the out-of-scope guard

## 1. Setup
Upload the project folder as a zip (`banking-agent.zip`) to `/content`, or clone it from your own GitHub repository.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

# Look for the zip in My Drive first, then on Colab's local disk
candidates = ["/content/drive/MyDrive/banking-agent.zip", "/content/banking-agent.zip"]
ZIP = next((p for p in candidates if os.path.isfile(p)), None)

if ZIP is None:
    raise FileNotFoundError("banking-agent.zip not found. Files in My Drive: "
                            + str(os.listdir("/content/drive/MyDrive")))
print("Using:", ZIP)

Mounted at /content/drive
Using: /content/drive/MyDrive/banking-agent.zip


In [ ]:
import os, zipfile, glob

!rm -rf /content/banking-agent
zipfile.ZipFile(ZIP).extractall("/content/banking-agent_unzip")

# Find the folder that contains src/ (works whether or not the zip has a top-level banking-agent folder)
src_dirs = glob.glob("/content/banking-agent_unzip/**/src", recursive=True)
assert src_dirs, "No src/ folder inside the zip"
PROJECT = os.path.dirname(src_dirs[0])
!mv "{PROJECT}" /content/banking-agent
%cd /content/banking-agent

!ls src                       # must list augment.py, gan.py, vae.py, peft_lora.py ...
!pip -q install transformers peft accelerate jsonschema scikit-learn scipy pandas gradio
# optional, only for QLoRA (4-bit): !pip -q install bitsandbytes
os.makedirs("evaluation/results", exist_ok=True)

/content/banking-agent
augment.py   decoding.py  guard.py  peft_lora.py  retrieval.py	 text_vocab.py
baseline.py  gan.py	  llm.py    prompts.py	  structured.py  vae.py


In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

2.11.0+cu130 True NVIDIA L4


## 2. VAE + GAN
Trains the conditional text VAE and the latent GAN, generates synthetic messages, filters them, and measures the effect on a classifier (full data and 10 % low-resource). Saves `models/vae.pt`, `models/gan.pt`, `data/synthetic_*.csv`, `evaluation/results/vae_gan_results.json`.

In [ ]:
!python -m src.augment --vae-epochs 30 --gan-steps 3000 --per-intent 30 --low-seeds 3

09:35:12 augment: == Full data: 10003 training messages
09:35:22 vae: VAE epoch 1/30 beta=0.03 recon=61.73 nats/sentence KL=21.76
09:35:27 vae: VAE epoch 5/30 beta=0.17 recon=31.34 nats/sentence KL=24.80
09:35:34 vae: VAE epoch 10/30 beta=0.33 recon=25.74 nats/sentence KL=18.41
09:35:40 vae: VAE epoch 15/30 beta=0.50 recon=23.85 nats/sentence KL=14.73
09:35:46 vae: VAE epoch 20/30 beta=0.50 recon=22.06 nats/sentence KL=14.58
09:35:53 vae: VAE epoch 25/30 beta=0.50 recon=20.62 nats/sentence KL=14.75
09:35:59 vae: VAE epoch 30/30 beta=0.50 recon=19.27 nats/sentence KL=14.86
09:36:00 gan: GAN step 1/3000  critic loss 7.262  generator loss -0.042
09:36:14 gan: GAN step 500/3000  critic loss -3.103  generator loss 2.578
09:36:28 gan: GAN step 1000/3000  critic loss -1.431  generator loss 1.803
09:36:42 gan: GAN step 1500/3000  critic loss -0.947  generator loss 1.983
09:36:56 gan: GAN step 2000/3000  critic loss -0.543  generator loss 2.901
09:37:10 gan: GAN step 2500/3000  critic loss -0.0

In [ ]:
import json, os
import pandas as pd

R = json.load(open("evaluation/results/vae_gan_results.json"))
print("keys in vae_gan_results.json:", list(R))

wanted = ["gen_stats", "latent_frechet", "kept_after_filter", "downstream_full", "knn5_vote", "guard"]
print(json.dumps({k: R[k] for k in wanted if k in R}, indent=2))
missing = [k for k in wanted if k not in R]
if missing:
    print("not in this file:", missing)

# Guard results are written to a separate file
guard_file = "evaluation/results/guard_eval.json"
if os.path.exists(guard_file):
    print("\nguard:", json.dumps(json.load(open(guard_file)), indent=2))
else:
    print(f"\n{guard_file} not found: run the guard script first (e.g. !python -m src.guard)")

print(pd.read_csv("evaluation/results/samples_vae.csv").head(10).to_string())
print(pd.read_csv("evaluation/results/samples_gan.csv").head(10).to_string())

keys in vae_gan_results.json: ['args', 'vae_history', 'gan_history', 'reference_lr', 'gen_stats', 'latent_frechet', 'kept_after_filter', 'downstream_full', 'knn5_vote', 'low_resource', 'low_resource_mean', 'low_resource_std', 'seconds']
{
  "gen_stats": {
    "vae": {
      "n": 2310,
      "unique_pct": 99.82683982683983,
      "novel_pct": 98.44155844155844,
      "distinct1": 0.033356574070676305,
      "distinct2": 0.25131309891343573,
      "mean_words": 11.796969696969697,
      "intent_fidelity_pct": 57.532467532467535,
      "mean_validator_conf": 0.3153856892947538
    },
    "gan": {
      "n": 2310,
      "unique_pct": 99.6969696969697,
      "novel_pct": 98.44155844155844,
      "distinct1": 0.033357374684457264,
      "distinct2": 0.2503147128245476,
      "mean_words": 12.004329004329005,
      "intent_fidelity_pct": 61.298701298701296,
      "mean_validator_conf": 0.33297541396336566
    }
  },
  "latent_frechet": {
    "prior_N(0,I)_vs_real": 1.0268880251608015,
    "ga

## 3. LoRA fine-tuning (PEFT)
Trains adapters on `data/train.csv` only. One epoch of 10,003 rows at batch 16 is about 626 steps. The adapter is saved to `models/lora-intent`; the base model stays frozen.

In [ ]:
!pip -q uninstall -y torchao

In [ ]:
!python -m src.peft_lora train --model Qwen/Qwen2.5-1.5B-Instruct --out models/lora-intent --epochs 1 --batch-size 16 --lr 2e-4 --r 16 --alpha 32

09:48:07 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
09:48:08 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
09:48:08 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
09:48:09 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
09:48:09 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
Loading weights: 100% 338/338 [00:00<00:00, 4249.23it/s]
09:48:12 peft: LoRA r=16 alpha=32: 18,464,768 trainable parameters of 1,562,179,072 (1.18%)
09:48:16 peft: step 1/626 epoch 1 loss 4.5109 lr 1.28e-05 (1s)
09:48:28 peft: step 25/626 epoch 1 loss 1.5505 lr 1.65e-04 (13s)
09:48:40 peft: step 50/626 epoch 1 loss 0.4141 l

In [ ]:
import json; print(json.dumps(json.load(open('evaluation/results/peft_summary.json')), indent=2))
print(json.load(open('models/lora-intent/train_log.json'))['trainable'], 'trainable parameters')

{
  "dev": {
    "accuracy": 0.93,
    "no_valid_intent": 0,
    "n": 100
  },
  "heldout": {
    "accuracy": 0.89,
    "no_valid_intent": 0,
    "n": 100
  }
}
18464768 trainable parameters


## 4. Compare the classifiers on the 100 held-out rows
RAG (default), PEFT, and the hybrid (PEFT first, RAG fallback). Each run is a separate process because `src/llm.py` loads the model at import time.

In [ ]:
!CLASSIFIER=retrieval python -m evaluation.final_eval
!ADAPTER_PATH=models/lora-intent CLASSIFIER=peft python -m evaluation.final_eval
!ADAPTER_PATH=models/lora-intent CLASSIFIER=peft+retrieval python -m evaluation.final_eval

09:53:45 llm: loading Qwen/Qwen2.5-1.5B-Instruct on cuda
09:53:46 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
09:53:46 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
09:53:46 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
09:53:47 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
09:53:47 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
Loading weights: 100% 338/338 [00:00<00:00, 5553.15it/s]
09:53:49 llm: model ready, dtype torch.bfloat16, 1.5B parameters
09:53:49 baseline: loaded 77 intent names from the BANKING77 train split
09:53:49 llm: LLM call: 549 prompt tokens, decoding={'do_sample': False}


In [ ]:
import pandas as pd
for f in ['results_heldout','results_heldout_peft','results_heldout_peft+retrieval']:
    d = pd.read_csv(f'evaluation/results/{f}.csv'); print(f, 'accuracy', d['correct'].mean(), 'no valid intent', int((d['predicted'].isna() | (d['predicted']=='')).sum()))

results_heldout accuracy 0.76 no valid intent 5
results_heldout_peft accuracy 0.89 no valid intent 0
results_heldout_peft+retrieval accuracy 0.89 no valid intent 0


## 5. Optional: RAG with the synthetic pool, and the VAE guard

In [ ]:
!RETRIEVAL_POOL=vae+gan python -m src.retrieval

09:56:06 llm: loading Qwen/Qwen2.5-1.5B-Instruct on cuda
09:56:06 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
09:56:06 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
09:56:07 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
09:56:07 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
09:56:07 httpx: HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
Loading weights: 100% 338/338 [00:00<00:00, 5808.19it/s]
09:56:09 llm: model ready, dtype torch.bfloat16, 1.5B parameters
09:56:09 baseline: loaded 77 intent names from the BANKING77 train split
09:56:10 llm: LLM call: 528 prompt tokens, decoding={'do_sample': False}


In [ ]:
!python - <<'EOF'
from src.guard import vae_score
for m in ['I lost my card', 'What is the weather in Paris?', 'How long does a transfer take?', 'Write me a poem']:
    print(m, vae_score(m))
EOF

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `EOF')
I lost my card ({'elbo': 2.911691188812256, 'unk': 0.0}, {'elbo': inf, 'unk': 1.0})
What is the weather in Paris? ({'elbo': 3.034986734390259, 'unk': 0.25}, {'elbo': inf, 'unk': 1.0})
How long does a transfer take? ({'elbo': 1.5463831424713135, 'unk': 0.0}, {'elbo': inf, 'unk': 1.0})
Write me a poem ({'elbo': 4.4133806228637695, 'unk': 0.4000000059604645}, {'elbo': inf, 'unk': 1.0})


NameError: name 'EOF' is not defined